# Notebook 0 — Exploratory Data Analysis (EDA)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/indicium15/ml-workshop/blob/main/workshop-2/notebooks/00_EDA_Exploratory_Data_Analysis.ipynb)

Use this notebook before modelling to understand any CSV dataset. It is designed for Google Colab or local Jupyter use: load a CSV, choose columns with GUI controls, and generate an exploratory report without editing code.

---

## Running This Notebook

You can use this notebook in either:

- **Google Colab**: best if you do not have Python installed.
- **Local Jupyter**: best if you already have the workshop folder on your computer.

### In Google Colab

1. Open this notebook from GitHub using the **Open in Colab** button.
2. Run the first setup cell.
3. Wait until it says the libraries and workshop files are loaded.
4. In **Step 1**, either use the pre-loaded sample dataset or upload your own CSV file.
5. Continue running each cell from top to bottom.

If a widget does not appear immediately in Colab, wait for the setup cell to finish, then rerun the current cell.

In [4]:
# SETUP
import sys, os, subprocess, importlib.util, tempfile
from pathlib import Path

# Works locally from workshop-2/notebooks, from workshop-2, or in Google Colab
# after opening the notebook from GitHub.
REPO_URL = 'https://github.com/indicium15/ml-workshop.git'

def _find_workshop_root():
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd() / 'workshop-2',
        Path.cwd().parent / 'workshop-2',
        Path('/content/ml-workshop/workshop-2'),
        Path('/content/workshop-2'),
    ]
    for candidate in candidates:
        if (candidate / 'utils' / 'data_loader.py').exists():
            return candidate.resolve()
    return None

IN_COLAB = 'google.colab' in sys.modules
WORKSHOP_ROOT = _find_workshop_root()

if WORKSHOP_ROOT is None and IN_COLAB:
    print('Workshop files not found in Colab. Cloning the workshop repository...')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, '/content/ml-workshop'], check=True)
    WORKSHOP_ROOT = _find_workshop_root()

if WORKSHOP_ROOT is None:
    raise FileNotFoundError(
        'Could not find workshop-2/utils. Locally, start Jupyter from the workshop-2 folder. '
        'In Colab, upload the workshop-2 folder or open the notebook from the GitHub repository.'
    )

if str(WORKSHOP_ROOT) not in sys.path:
    sys.path.insert(0, str(WORKSHOP_ROOT))

required_modules = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'sklearn': 'scikit-learn',
    'scipy': 'scipy',
    'matplotlib': 'matplotlib',
    'seaborn': 'seaborn',
    'ipywidgets': 'ipywidgets',
}
missing = [pip_name for module_name, pip_name in required_modules.items()
           if importlib.util.find_spec(module_name) is None]
if missing:
    print('Installing missing packages:', ', '.join(missing))
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(WORKSHOP_ROOT / 'requirements.txt')], check=True)

import ipywidgets as widgets

print(f'Using workshop files from: {WORKSHOP_ROOT}')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from IPython.display import display

# Keep Matplotlib/font caches in a writable temp folder. This avoids long first-plot stalls
# in locked-down notebook environments where the default home cache is not writable.
_cache_root = Path(tempfile.gettempdir()) / 'ml_workshop_cache'
_mpl_cache = _cache_root / 'matplotlib'
_mpl_cache.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(_mpl_cache))
os.environ.setdefault('XDG_CACHE_HOME', str(_cache_root))

import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
import matplotlib.pyplot as plt
%matplotlib inline

import seaborn as sns

from utils.data_loader import DataLoaderWidget
from utils.plotting import plot_correlation_heatmap, plot_feature_distributions

print('Libraries loaded ✓')


Using workshop files from: /Users/chaitanya/Documents/Coding/ml-workshop/workshop-2
Libraries loaded ✓


### Colab Notes

- The first setup cell may take a minute because it checks for required packages.
- Uploaded files in Colab are temporary. If the runtime disconnects, upload the file again.
- To keep your own completed version, use **File → Save a copy in Drive**.
- If widgets do not display, run the setup cell again, then rerun the current cell.

## Step 1 — Load Data

The sample dataset is already available when you run the setup cell.

To use your own data in Colab:

1. Click the upload control.
2. Choose a `.csv` file from your computer.
3. Wait for the preview table to appear.
4. Select the columns you want to use.
5. Click **Confirm Selection**.

For local Jupyter, you can either upload a CSV or type a path to a CSV file on your machine.

### What the Load Data controls do

| Control | What it does | When to adjust it |
|---|---|---|
| **CSV path** | Points the notebook to a CSV file on disk. The default path loads the workshop sample dataset. | Change this when running locally with your own CSV file. |
| **Load CSV** | Reads the file from **CSV path** and displays a short preview plus column counts. | Click this after editing the path. |
| **Or upload** | Lets you upload a CSV directly in Colab/Jupyter instead of typing a path. | Use this when your CSV is on your computer rather than already in the notebook folder. |
| **Features** | Selects the input columns the later analysis/model will use. Hold Ctrl/Cmd to select multiple columns. | Exclude IDs, labels/targets, free-text notes, or columns you do not want the method to learn from. |
| **Select all** | Selects every column except common ID/target columns. | Useful as a starting point, then remove anything inappropriate. |
| **Numeric only** | Selects only integer/float columns. | Useful for a simpler first run or for methods where you want to avoid categorical encoding. |
| **Confirm Selection** | Saves the selected features for later cells. | Click this before preprocessing or running any analysis. |

> Keep a copy of your original dataset unchanged. These notebooks do not edit the source CSV.

Select the columns you want to explore, then click **Confirm Selection**. You can always rerun this cell and choose different columns if your first selection is too broad or too narrow.


In [ ]:
# LOAD DATA
loader = DataLoaderWidget(show_label_selector=False)
loader.display()


## Step 2 — Configure and Run EDA

Click **Refresh Columns** after confirming your data. Then choose the columns, preview size, missing-value threshold, correlation method, and optional grouping column.

### What the EDA controls do

| Control | What it does | When to adjust it |
|---|---|---|
| **Refresh Columns** | Updates the control choices from the columns you confirmed in Step 1. | Click this after loading new data or changing the selected feature columns. |
| **Run EDA Report** | Generates the snapshot, column overview, missing-value report, summaries, distributions, correlation heatmap, and optional group comparison. | Click this whenever you change an EDA setting. |
| **Preview rows** | Controls how many first rows are shown in the Dataset Snapshot using `df.head(...)`. | Increase it when you want to inspect more examples; decrease it to keep the output compact. It does not change the analysis. |
| **Missing >=** | Sets the missing-value percentage cutoff for the missing-values section. | Lower it to catch columns with small amounts of missing data; raise it to focus only on serious missingness. |
| **Max categories** | Limits how many category labels are shown in categorical plots/summaries. | Lower it when charts are crowded; raise it when a categorical column has many meaningful levels. |
| **Correlation** | Selects the correlation method for numeric columns: Pearson, Spearman, or Kendall. | Use Pearson for linear relationships, Spearman for rank/monotonic relationships, and Kendall for smaller samples or ordinal-style data. |
| **Show distributions** | Toggles histograms/count plots for selected columns. | Turn it off when you only need summaries or want faster output. |
| **Show correlation heatmap** | Toggles the numeric correlation heatmap. | Turn it on when comparing relationships among numeric features; it needs at least two numeric selected columns. |
| **Group by** | Chooses an optional categorical column for comparing numeric feature means by group. | Use it to compare groups such as performance bands, departments, classes, or cohorts. |
| **Plot columns** | Chooses which columns are included in the EDA report and plots. | Select fewer columns for focused analysis or faster, cleaner charts. |



In [ ]:
# EDA CONTROLS
_eda_out = widgets.Output()

_preview_rows = widgets.IntSlider(
    value=5, min=3, max=20, step=1,
    description='Preview rows:',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='430px'),
)
_missing_threshold = widgets.FloatSlider(
    value=0.0, min=0.0, max=1.0, step=0.05,
    description='Missing >=',
    readout_format='.0%',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='430px'),
)
_max_categories = widgets.IntSlider(
    value=12, min=5, max=30, step=1,
    description='Max categories:',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='430px'),
)
_plot_cols = widgets.SelectMultiple(
    options=[],
    value=(),
    description='Plot columns:',
    style={'description_width': '110px'},
    layout=widgets.Layout(width='640px', height='220px'),
)
_group_col = widgets.Dropdown(
    options=['(none)'],
    value='(none)',
    description='Group by:',
    style={'description_width': '110px'},
    layout=widgets.Layout(width='430px'),
)
_corr_method = widgets.Dropdown(
    options=['pearson', 'spearman', 'kendall'],
    value='pearson',
    description='Correlation:',
    style={'description_width': '110px'},
    layout=widgets.Layout(width='300px'),
)
_show_corr = widgets.Checkbox(
    value=True,
    description='Show correlation heatmap',
    layout=widgets.Layout(width='260px'),
)
_show_distributions = widgets.Checkbox(
    value=True,
    description='Show distributions',
    layout=widgets.Layout(width='220px'),
)
_refresh_cols_btn = widgets.Button(
    description='Refresh Columns',
    button_style='',
    icon='refresh',
)
_run_eda_btn = widgets.Button(
    description='Run EDA Report',
    button_style='primary',
    icon='bar-chart',
)

_MAX_TABLE_ROWS = 80
_MAX_PLOT_ROWS = 5000
_MAX_CORR_ROWS = 5000

def _active_df():
    if loader.X_df is not None:
        return loader.X_df.copy()
    if loader.df is not None:
        return loader.df.copy()
    return None

def _refresh_eda_columns(_btn=None):
    df = _active_df()
    _eda_out.clear_output()
    if df is None:
        _plot_cols.options = []
        _plot_cols.value = ()
        _group_col.options = ['(none)']
        _group_col.value = '(none)'
        with _eda_out:
            display(widgets.HTML('<span style="color:red">Load and confirm data first.</span>'))
        return

    cols = list(df.columns)
    numeric_cols = list(df.select_dtypes(include='number').columns)
    low_card_cols = [c for c in cols if df[c].nunique(dropna=True) <= _max_categories.value]
    default_cols = (numeric_cols[:8] if numeric_cols else cols[:8])

    _plot_cols.options = cols
    _plot_cols.value = tuple(default_cols)
    group_options = ['(none)'] + low_card_cols
    _group_col.options = group_options
    _group_col.value = '(none)'

    with _eda_out:
        display(widgets.HTML(
            f'<span style="color:green">✓ Ready: {len(df):,} rows × {len(cols)} selected columns. '
            f'{len(numeric_cols)} numeric, {len(cols) - len(numeric_cols)} non-numeric.</span>'
        ))

def _column_overview(df):
    overview = pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'non_null': df.notna().sum(),
        'missing': df.isna().sum(),
        'missing_pct': df.isna().mean(),
        'unique_values': df.nunique(dropna=True),
    })
    return overview.sort_values(['missing_pct', 'unique_values'], ascending=[False, False])

def _limited_display_df(df, max_rows=_MAX_TABLE_ROWS):
    if len(df) <= max_rows:
        return df
    display(widgets.HTML(
        f'<span style="color:#555">Showing first {max_rows} rows of {len(df):,} to keep the notebook responsive.</span>'
    ))
    return df.head(max_rows)

def _plot_eda_distributions(df, columns, ncols=3, max_rows=_MAX_PLOT_ROWS):
    columns = [c for c in columns if c in df.columns]
    if not columns:
        return None
    plot_df = df.sample(max_rows, random_state=42) if len(df) > max_rows else df
    if len(df) > max_rows:
        display(widgets.HTML(
            f'<span style="color:#555">Distribution plots use a reproducible sample of {max_rows:,} rows for speed.</span>'
        ))

    nrows = int(np.ceil(len(columns) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3 * nrows))
    axes = np.array(axes).reshape(-1)
    for i, col in enumerate(columns):
        ax = axes[i]
        series = plot_df[col]
        if pd.api.types.is_numeric_dtype(series) and series.nunique(dropna=True) > 10:
            ax.hist(series.dropna(), bins=20, color='steelblue', edgecolor='white')
        else:
            counts = series.astype('string').fillna('(missing)').value_counts(dropna=False).head(_max_categories.value)
            counts.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
            ax.tick_params(axis='x', labelrotation=45)
        ax.set_title(col, fontsize=10)
        ax.set_xlabel('')
    for ax in axes[len(columns):]:
        ax.set_visible(False)
    plt.tight_layout()
    return fig

def _run_eda(_btn):
    _eda_out.clear_output()
    with _eda_out:
        try:
            _run_eda_btn.disabled = True
            _refresh_cols_btn.disabled = True
            df = _active_df()
            if df is None:
                display(widgets.HTML('<span style="color:red">Load and confirm data first.</span>'))
                _run_eda_btn.disabled = False
                _refresh_cols_btn.disabled = False
                return
            selected_cols = list(_plot_cols.value) or list(df.columns)[:8]
            selected_cols = [c for c in selected_cols if c in df.columns]
            report_df = df[selected_cols].copy()

            display(widgets.HTML(f'<h3>Dataset Snapshot</h3><b>{len(df):,}</b> rows × <b>{len(df.columns):,}</b> selected columns'))
            display(df.head(_preview_rows.value))

            display(widgets.HTML('<h3>Column Overview</h3>'))
            overview = _column_overview(df)
            overview_display = _limited_display_df(overview)
            display(overview_display.style.format({'missing_pct': '{:.1%}'}).background_gradient(subset=['missing_pct'], cmap='Reds'))

            missing = overview[overview['missing_pct'] >= _missing_threshold.value]
            display(widgets.HTML(f'<h3>Missing Values ≥ {_missing_threshold.value:.0%}</h3>'))
            if missing.empty:
                display(widgets.HTML('<span style="color:green">No columns meet this missing-value threshold.</span>'))
            else:
                missing_display = _limited_display_df(missing[['missing', 'missing_pct']])
                display(missing_display.style.format({'missing_pct': '{:.1%}'}).background_gradient(cmap='Reds'))

            numeric_cols = list(report_df.select_dtypes(include='number').columns)
            cat_cols = [c for c in report_df.columns if c not in numeric_cols]

            if numeric_cols:
                display(widgets.HTML('<h3>Numeric Summary</h3>'))
                display(report_df[numeric_cols].describe().T.style.format('{:.2f}'))
            if cat_cols:
                display(widgets.HTML('<h3>Categorical Summary</h3>'))
                cat_summary = []
                for col in cat_cols:
                    counts = report_df[col].astype(str).value_counts(dropna=False).head(_max_categories.value)
                    cat_summary.append({
                        'column': col,
                        'unique_values': report_df[col].nunique(dropna=True),
                        'top_value': counts.index[0] if len(counts) else '',
                        'top_count': int(counts.iloc[0]) if len(counts) else 0,
                    })
                display(pd.DataFrame(cat_summary))

            if _show_distributions.value and selected_cols:
                display(widgets.HTML('<h3>Distributions</h3>'))
                fig = _plot_eda_distributions(report_df, selected_cols, ncols=3)
                plt.show()
                plt.close(fig)

            if _show_corr.value and len(numeric_cols) >= 2:
                display(widgets.HTML(f'<h3>Correlation Heatmap ({_corr_method.value})</h3>'))
                corr_source = report_df[numeric_cols]
                if len(corr_source) > _MAX_CORR_ROWS:
                    corr_source = corr_source.sample(_MAX_CORR_ROWS, random_state=42)
                    display(widgets.HTML(
                        f'<span style="color:#555">Correlation uses a reproducible sample of {_MAX_CORR_ROWS:,} rows for speed.</span>'
                    ))
                corr_df = corr_source.corr(method=_corr_method.value)
                fig, ax = plt.subplots(figsize=(max(7, len(numeric_cols) * 0.75), max(5, len(numeric_cols) * 0.55)))
                mask = np.triu(np.ones_like(corr_df, dtype=bool))
                sns.heatmap(corr_df, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn', center=0, linewidths=0.5, ax=ax)
                ax.set_title(f'{_corr_method.value.title()} Correlation')
                plt.tight_layout()
                plt.show()
                plt.close(fig)
            elif _show_corr.value:
                display(widgets.HTML('<span style="color:#555">Correlation heatmap needs at least two numeric selected columns.</span>'))

            group_col = _group_col.value
            if group_col != '(none)' and group_col in df.columns and numeric_cols:
                display(widgets.HTML(f'<h3>Group Comparison by <code>{group_col}</code></h3>'))
                grouped = df[[group_col] + numeric_cols].copy()
                if grouped[group_col].nunique(dropna=True) > _max_categories.value:
                    top_groups = grouped[group_col].astype(str).value_counts().head(_max_categories.value).index
                    grouped = grouped[grouped[group_col].astype(str).isin(top_groups)]
                means = grouped.groupby(group_col, dropna=False)[numeric_cols].mean()
                display(means.style.format('{:.2f}').background_gradient(cmap='YlGnBu', axis=0))
                fig, ax = plt.subplots(figsize=(max(8, len(numeric_cols) * 0.7), max(4, len(means) * 0.55)))
                sns.heatmap(means, annot=True, fmt='.1f', cmap='YlGnBu', linewidths=0.5, ax=ax)
                ax.set_title(f'Mean numeric values by {group_col}')
                plt.tight_layout()
                plt.show()
                plt.close(fig)

            _run_eda_btn.disabled = False
            _refresh_cols_btn.disabled = False

        except Exception as exc:
            _run_eda_btn.disabled = False
            _refresh_cols_btn.disabled = False
            import traceback
            display(widgets.HTML(f'<span style="color:red">Error: {exc}<br><pre>{traceback.format_exc()}</pre></span>'))

_refresh_cols_btn.on_click(_refresh_eda_columns)
_run_eda_btn.on_click(_run_eda)

display(widgets.VBox([
    widgets.HTML('<h3>🔎 EDA Controls</h3>'),
    widgets.HBox([_refresh_cols_btn, _run_eda_btn]),
    widgets.HBox([_preview_rows, _missing_threshold]),
    widgets.HBox([_max_categories, _corr_method]),
    widgets.HBox([_show_distributions, _show_corr]),
    _group_col,
    _plot_cols,
    _eda_out,
]))


## After the Workshop: Reusing This EDA Notebook

Use this notebook whenever you receive a new CSV dataset and want to quickly understand it before modelling.

Recommended workflow:

1. Load the CSV.
2. Check missing values and unusual columns.
3. Inspect numeric distributions.
4. Compare groups if you have a useful category column.
5. Identify columns to exclude before clustering or prediction.

### What to Save

For your own notes or report, save:

- the dataset shape: number of rows and columns
- columns with high missingness
- strongly correlated variables
- unusual distributions or outliers
- candidate columns for modelling
- columns that should be excluded because they are IDs, labels, or leakage risks